In [17]:
import numpy as np
import pandas as pd
import scipy as sp

In [1]:
import requests, pandas as pd, time

SITE_SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
SITE_SUMMARY   = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def j(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json()

def event_ids_from_scoreboard(year: int, seasontype: int, week: int):
    url = f"{SITE_SCOREBOARD}?dates={year}&seasontype={seasontype}&week={week}"
    data = j(url)
    return [e["id"] for e in data.get("events", [])], url

def _statmap_from_entry(cat: dict, athlete_block: dict) -> dict:
    """
    cat: a category object (e.g., passing/rushing/defensive)
    athlete_block: one athlete entry under that category
    Returns flat {stat_name: value} handling multiple ESPN shapes.
    """
    # 1) Newer shape: list of dicts with name/value/displayValue
    stats = athlete_block.get("stats")
    if isinstance(stats, list) and stats and isinstance(stats[0], dict):
        out = {}
        for s in stats:
            key = s.get("name")
            val = s.get("displayValue")
            if val is None:
                val = s.get("value")
            if key is not None:
                out[key] = val
        if out:
            return out

    # 2) Older/common shape: stats are list of strings; labels live on cat or athlete block
    labels = (athlete_block.get("labels")
              or cat.get("labels")
              or cat.get("names"))  # some payloads use "names" instead of "labels"
    totals = (athlete_block.get("stats")
              or athlete_block.get("totals"))
    if isinstance(labels, list) and isinstance(totals, list) and len(labels) == len(totals):
        return {lbl: val for lbl, val in zip(labels, totals)}

    # 3) Fallback: empty
    return {}

def player_boxscores_for_week(year: int, seasontype: int, week: int) -> pd.DataFrame:
    eids, used_url = event_ids_from_scoreboard(year, seasontype, week)
    print(f"Found {len(eids)} games via: {used_url}")
    rows = []

    for eid in eids:
        data = j(f"{SITE_SUMMARY}?event={eid}")
        box = data.get("boxscore", {})

        # Player stats live under box['players']; each entry == one team’s players by category
        for entry in box.get("players", []):
            team_info = entry.get("team") or {}
            team_name = (team_info.get("displayName")
                         or team_info.get("name")
                         or team_info.get("abbreviation"))
            for cat in entry.get("statistics", []):   # passing/rushing/receiving/defensive/kicking/returns, etc.
                stype = cat.get("type") or cat.get("name")
                for p in cat.get("athletes", []):
                    a = p.get("athlete", {}) or {}
                    statmap = _statmap_from_entry(cat, p)
                    base = {
                        "season": year, "seasontype": seasontype, "week": week,
                        "event_id": eid, "team": team_name, "stat_type": stype,
                        "athlete_id": a.get("id"), "player": a.get("displayName"),
                    }
                    rows.append({**base, **statmap})
        time.sleep(0.12)  # be polite

    df = pd.DataFrame(rows)
    print("Rows:", len(df))
    if not df.empty:
        # show a compact peek
        keep = ["season","week","team","stat_type","player","athlete_id"]
        cols = keep + [c for c in df.columns if c not in keep][:8]  # first few stat cols
        print(df[cols].head(20).to_string(index=False))
    else:
        print("Empty — try another (year, week) or open the printed scoreboard URL to confirm games.")
    return df

# 🔽 Try this
df = player_boxscores_for_week(2024, 2, 1)   # 2024 regular season, week 1


Found 16 games via: https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=2024&seasontype=2&week=1
Rows: 1228
 season  week             team stat_type           player athlete_id  seasontype  event_id C/ATT YDS  AVG  TD INT SACKS
   2024     1 Baltimore Ravens   passing    Lamar Jackson    3916387           2 401671789 26/41 273  6.7   1   0   1-6
   2024     1 Baltimore Ravens   rushing    Lamar Jackson    3916387           2 401671789   NaN 122  7.6   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing    Derrick Henry    3043078           2 401671789   NaN  46  3.5   1 NaN   NaN
   2024     1 Baltimore Ravens   rushing      Zay Flowers    4429615           2 401671789   NaN  14  7.0   0 NaN   NaN
   2024     1 Baltimore Ravens   rushing     Justice Hill    4038441           2 401671789   NaN   3  3.0   0 NaN   NaN
   2024     1 Baltimore Ravens receiving    Isaiah Likely    4361050           2 401671789   NaN 111 12.3   1 NaN   NaN
   2024     1 Baltimore Rave

In [ ]:
import requests, pandas as pd, time

# --- ESPN endpoints ---
SITE_SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"
SITE_SUMMARY    = "https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary"

def j(url):
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json()

def event_ids_for_date(yyyymmdd: str):
    """Return ESPN event IDs for one calendar date (e.g., '20251019')."""
    url = f"{SITE_SCOREBOARD}?dates={yyyymmdd}"
    data = j(url)
    return [e["id"] for e in data.get("events", [])], url

# --- parsing helpers (handles ESPN payload variants) ---
def _statmap_from_player(cat: dict, athlete_block: dict) -> dict:
    # case A: list of dicts with {name, value/displayValue}
    stats = athlete_block.get("stats")
    if isinstance(stats, list) and stats and isinstance(stats[0], dict):
        out = {}
        for s in stats:
            key = s.get("name") or s.get("label")
            val = s.get("displayValue", s.get("value"))
            if key is not None:
                out[key] = val
        if out: return out
    # case B: parallel labels + totals
    labels = athlete_block.get("labels") or cat.get("labels") or cat.get("names")
    totals = athlete_block.get("stats")  or athlete_block.get("totals")
    if isinstance(labels, list) and isinstance(totals, list) and len(labels) == len(totals):
        return {lbl: val for lbl, val in zip(labels, totals)}
    return {}

def _statmap_from_team(team_block: dict) -> dict:
    out = {}
    for item in team_block.get("statistics", []):
        # item can be {'name': 'rushingYards', 'displayValue': '132'} or have nested lists
        key = item.get("name") or item.get("label")
        val = item.get("displayValue", item.get("value"))
        if key and val is not None:
            out[key] = val
        # nested case: {'labels': [...], 'displayValues': [...]}
        labels = item.get("labels"); vals = item.get("displayValues")
        if isinstance(labels, list) and isinstance(vals, list) and len(labels)==len(vals):
            out.update({lbl: v for lbl, v in zip(labels, vals)})
    return out

def pull_players_and_teams_for_date(yyyymmdd: str):
    eids, used_url = event_ids_for_date(yyyymmdd)
    print(f"Date {yyyymmdd}: found {len(eids)} games via {used_url}")
    player_rows, team_rows = [], []

    for eid in eids:
        data = j(f"{SITE_SUMMARY}?event={eid}")
        box = data.get("boxscore", {})

        # --- players ---
        for entry in box.get("players", []):     # one block per team
            team_info = entry.get("team") or {}
            team_name = (team_info.get("displayName") or
                         team_info.get("name") or
                         team_info.get("abbreviation"))
            for cat in entry.get("statistics", []):   # passing/rushing/receiving/defensive/…
                stype = cat.get("type") or cat.get("name")
                for p in cat.get("athletes", []):
                    a = p.get("athlete", {}) or {}
                    statmap = _statmap_from_player(cat, p)
                    base = {
                        "event_id": eid,
                        "team": team_name,
                        "stat_type": stype,
                        "athlete_id": a.get("id"),
                        "player": a.get("displayName"),
                    }
                    player_rows.append({**base, **statmap})

        # --- team totals (optional but handy) ---
        for t in box.get("teams", []):
            tinfo = t.get("team", {}) or {}
            team_name = (tinfo.get("displayName") or
                         tinfo.get("name") or
                         tinfo.get("abbreviation"))
            tstats = _statmap_from_team(t)
            team_rows.append({"event_id": eid, "team": team_name, **tstats})

        time.sleep(0.1)  # be polite

    players_df = pd.DataFrame(player_rows)
    teams_df   = pd.DataFrame(team_rows)

    print(f"\nPlayers rows: {len(players_df)}")
    if not players_df.empty:
        keep = ["event_id","team","stat_type","player","athlete_id"]
        cols = keep + [c for c in players_df.columns if c not in keep][:8]
        print(players_df[cols].head(20).to_string(index=False))

    print(f"\nTeams rows: {len(teams_df)}")
    if not teams_df.empty:
        print(teams_df.head(10).to_string(index=False))

    return players_df, teams_df

# 🔽 Run for last Sunday (10/19/2025)
players_df, teams_df = pull_players_and_teams_for_date("20251019")

# Optionally save
players_df.to_csv("nfl_players_20251019.csv", index=False)
teams_df.to_csv("nfl_teams_20251019.csv", index=False)
print("\nSaved: nfl_players_20251019.csv  &  nfl_teams_20251019.csv")




Date 20251019: found 12 games via https://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates=20251019

Players rows: 968
 event_id             team stat_type              player athlete_id C/ATT YDS  AVG TD INT SACKS  QBR   RTG
401772635 Los Angeles Rams   passing    Matthew Stafford      12483 21/33 182  5.5  5   0   0-0 76.9 117.7
401772635 Los Angeles Rams   rushing      Kyren Williams    4430737   NaN  54  4.5  0 NaN   NaN  NaN   NaN
401772635 Los Angeles Rams   rushing         Blake Corum    4429096   NaN  37  3.1  0 NaN   NaN  NaN   NaN
401772635 Los Angeles Rams   rushing    Matthew Stafford      12483   NaN   1  0.5  0 NaN   NaN  NaN   NaN
401772635 Los Angeles Rams   rushing     Jimmy Garoppolo      16760   NaN  -3 -1.0  0 NaN   NaN  NaN   NaN
401772635 Los Angeles Rams receiving     Colby Parkinson    4242557   NaN  47 15.7  0 NaN   NaN  NaN   NaN
401772635 Los Angeles Rams receiving       Davante Adams      16800   NaN  35  7.0  3 NaN   NaN  NaN   NaN
401772

In [26]:
print(len(players_df.columns), "columns:")
print(players_df.columns.tolist())


31 columns:
['event_id', 'team', 'stat_type', 'athlete_id', 'player', 'C/ATT', 'YDS', 'AVG', 'TD', 'INT', 'SACKS', 'QBR', 'RTG', 'CAR', 'LONG', 'REC', 'TGTS', 'TOT', 'SOLO', 'TFL', 'PD', 'QB HTS', 'NO', 'FG', 'PCT', 'XP', 'PTS', 'TB', 'In 20', 'FUM', 'LOST']


In [29]:
def view_cat(df, cat, n=25):
    meta = ["event_id","team","player","athlete_id","stat_type"]
    sub = df[df["stat_type"]==cat].copy()
    # move meta to front, keep all stat columns after
    cols = meta + [c for c in sub.columns if c not in meta]
    return sub[cols].head(n)

print(view_cat(players_df, "passing", 25).to_string(index=False))
# try: "rushing", "receiving", "defensive", "kicking", "returns"


 event_id                  team           player athlete_id stat_type C/ATT YDS  AVG TD INT SACKS   QBR   RTG CAR LONG REC TGTS TOT SOLO TFL  PD QB HTS  NO  FG PCT  XP PTS  TB In 20 FUM LOST
401772635      Los Angeles Rams Matthew Stafford      12483   passing 21/33 182  5.5  5   0   0-0  76.9 117.7 NaN  NaN NaN  NaN NaN  NaN NaN NaN    NaN NaN NaN NaN NaN NaN NaN   NaN NaN  NaN
401772635  Jacksonville Jaguars  Trevor Lawrence    4360310   passing 23/48 296  6.2  1   0  7-32  13.0  74.7 NaN  NaN NaN  NaN NaN  NaN NaN NaN    NaN NaN NaN NaN NaN NaN NaN   NaN NaN  NaN
401772861    New Orleans Saints  Spencer Rattler    4426339   passing 20/32 233  7.3  2   3  4-24  30.1  66.3 NaN  NaN NaN  NaN NaN  NaN NaN NaN    NaN NaN NaN NaN NaN NaN NaN   NaN NaN  NaN
401772861         Chicago Bears   Caleb Williams    4431611   passing 15/26 172  6.6  0   1   1-8  19.5  61.7 NaN  NaN NaN  NaN NaN  NaN NaN NaN    NaN NaN NaN NaN NaN NaN NaN   NaN NaN  NaN
401772754        Miami Dolphins   Tua Tagovai